**Persiapan SparkSession**

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Tugas6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah.
Tampilkan jumlah baris dan `printSchema()` masing-masing.

In [5]:
# Membaca data transaksi
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

# Membaca data produk
df_produk = spark.read.json("tugas6_produk.json")

# Membaca data ulasan
df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

print("Data transaksi:")
df_transaksi.printSchema()

print("Data produk:")
df_produk.printSchema()

print("Data ulasan:")
df_ulasan.printSchema()

Data transaksi:
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data produk:
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data ulasan:
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [6]:
# pastikan datanya terbaca
print("Jumlah data transaksi:", df_transaksi.count())
print("Jumlah data produk:", df_produk.count())
print("Jumlah data ulasan:", df_ulasan.count())

Jumlah data transaksi: 5000
Jumlah data produk: 30
Jumlah data ulasan: 3500


**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan 
**`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan 
**`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid).
    Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

1. Gabungkan transaksi dengan ulasan

In [7]:
df_transaksi_ulasan = df_transaksi.join(
    df_ulasan,
    on="order_id",
    how="left"
)

df_transaksi_ulasan.show(10)

print("Jumlah transaksi setelah join ulasan:", df_transaksi_ulasan.count())

+--------+----------+------------+-------------------+------+
|order_id|product_id|unit_terjual|            tanggal|rating|
+--------+----------+------------+-------------------+------+
|     TX0|        21|           2|2026-10-10 00:00:00|     1|
|     TX1|         8|           6|2026-10-25 00:00:00|  NULL|
|     TX2|        25|           4|2026-10-13 00:00:00|     2|
|     TX3|         3|           4|2026-10-18 00:00:00|     5|
|     TX4|        19|           6|2026-10-07 00:00:00|  NULL|
|     TX5|        10|           3|2026-10-01 00:00:00|     5|
|     TX6|        26|           6|2026-10-24 00:00:00|     4|
|     TX7|         4|           1|2026-10-03 00:00:00|  NULL|
|     TX8|         7|           2|2026-10-09 00:00:00|     5|
|     TX9|        30|           1|2026-10-27 00:00:00|  NULL|
+--------+----------+------------+-------------------+------+
only showing top 10 rows

Jumlah transaksi setelah join ulasan: 5000


2. Gabungkan dengan data produk

In [8]:
df_etl = df_transaksi_ulasan.join(
    df_produk,
    on="product_id",
    how="inner"
)

df_etl.show(10)

df_etl.printSchema()

+----------+--------+------------+-------------------+------+------+------------+-----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|
+----------+--------+------------+-------------------+------+------+------------+-----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|
|        26|     TX6|           6|2026-10-24 00:00:00|     4| 75000|   Kesehatan|  Produk-26|
|         4|     TX7|           1|2026-10-03 00:00:00|  NULL

3. Tambahkan total pendapatan Sesuai soal: total_pendapatan = unit_terjual x harga

In [9]:
df_etl = df_etl.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

df_etl.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).


In [10]:
df_etl = df_etl.fillna({"rating": 0})

df_etl = df_etl.withColumn(
    "ada_ulasan",
    when(col("rating") > 0, True).otherwise(False)
)

df_etl.show(10)

print("Jumlah rating yang masih kosong:",
      df_etl.filter(col("rating").isNull()).count())
         
    

NameError: name 'when' is not defined

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`.
Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

In [ ]:
hdfs_path = "hdfs://localhost:9000/user/vey/tugas6/hasil_etl"

df_etl.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(hdfs_path)

print("Data berhasil disimpan ke HDFS")

!hdfs dfs -ls -R /user/vey/tugas6/hasil_etl

In [ ]:
df_final = spark.read.parquet(
    "hdfs://localhost:9000/user/vey/tugas6/hasil_etl"
)

print("Jumlah baris hasil akhir:", df_final.count())

df_final.show(10)

**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

In [ ]:
from pyspark.sql.functions import col, count, sum, round

# Menghitung jumlah transaksi dan transaksi yang memiliki ulasan
insight = df_final.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("transaksi_dengan_ulasan")
)

# Menghitung persentase transaksi yang memiliki ulasan
insight = insight.withColumn(
    "persentase_ulasan",
    round(col("transaksi_dengan_ulasan") / col("total_transaksi") * 100, 2)
)

insight.show()

**explore**

In [11]:
#  Deteksi Produk dengan Rating Sempurna (Extract & Filter Lanjutan)
from pyspark.sql.functions import col

# Filter hanya transaksi yang mendapatkan rating sempurna (Bintang 5)
df_rating_sempurna = df_etl.filter(col("rating") == 5)

print("=== EKSPLORASI 1: SAMPEL PRODUK DENGAN RATING BINTANG 5 ===")
df_rating_sempurna.select("order_id", "nama_produk", "kategori", "harga", "rating") \
                  .orderBy(col("harga").desc()) \
                  .show(5)

=== EKSPLORASI 1: SAMPEL PRODUK DENGAN RATING BINTANG 5 ===
+--------+-----------+----------+------+------+
|order_id|nama_produk|  kategori| harga|rating|
+--------+-----------+----------+------+------+
|   TX217|  Produk-13|   Fashion|250000|     5|
|   TX472|   Produk-2|Elektronik|250000|     5|
|   TX332|  Produk-13|   Fashion|250000|     5|
|   TX358|   Produk-2|Elektronik|250000|     5|
|   TX464|   Produk-2|Elektronik|250000|     5|
+--------+-----------+----------+------+------+
only showing top 5 rows



In [15]:
from pyspark.sql.functions import col, sum as spark_sum

# Meringkas jumlah unit terjual berdasarkan kelompok kategori
df_eksplorasi_unit = df_etl.groupBy("kategori").agg(
    spark_sum("unit_terjual").alias("total_unit_laku")
)

print("=== JUMLAH UNIT TERJUAL PER KATEGORI PRODUK ===")
df_eksplorasi_unit.orderBy(col("total_unit_laku").desc()).show()

=== JUMLAH UNIT TERJUAL PER KATEGORI PRODUK ===
+------------+---------------+
|    kategori|total_unit_laku|
+------------+---------------+
|  Elektronik|           6641|
|     Fashion|           4117|
|   Kesehatan|           4020|
|Rumah Tangga|           3241|
|     Makanan|           2172|
+------------+---------------+

